<a href="https://colab.research.google.com/github/minjaaaa/Schneider-Electric-Internship/blob/main/notebooks/04_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -q https://github.com/minjaaaa/Schneider-Electric-Internship.git repo
%cd repo
!pip install -q qdrant-client FlagEmbedding sentence-transformers python-dotenv bitsandbytes accelerate

/content/repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 63.6 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Pretraga: konfiguracija eksperimenta 8 (hibrid, top 5)

In [3]:
GEN_MODEL = "Qwen/Qwen2.5-14B-Instruct"
TOP_K = 5
RUN_NAME = "gen_hybrid_qwen14b"

In [4]:
import sys, json
sys.path.insert(0, "src")
from chunking import load_jsonl
from config import DATA_PROCESSED, EVAL_QUESTIONS
from figures import figure_chunks
from indexing import get_client, ensure_index, hybrid_search

questions = json.loads(EVAL_QUESTIONS.read_text(encoding="utf-8"))
chunks = load_jsonl(DATA_PROCESSED / "chunks_section.jsonl") + figure_chunks()
client = get_client(in_memory=True)                     # Qdrant u memoriji; u Colab-u nema qdrant_data/
collection = ensure_index(client, chunks, with_path=True, hybrid=True)

retrieved = {q["id"]: hybrid_search(client, collection, q["question"], TOP_K, using="hybrid",
                                    exclude_superseded=True)
             for q in questions}
print(len(chunks), "chunk-ova;", len(retrieved), "pitanja pretraženo")

Indeksiram m3_283c9320e9 (108 chunk-ova)...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 7/7 [00:03<00:00,  2.08it/s]


108 chunk-ova; 49 pitanja pretraženo


In [5]:
from generate import HFChatModel, answer_question
llm = HFChatModel(GEN_MODEL)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Probni odgovor na Q02 (memo menja pravilo)

In [6]:
q = next(q for q in questions if q["id"] == "Q02")      # memo menja pravilo: test pravila o statusu
r = answer_question(q["question"], retrieved["Q02"], llm)
print(r["raw"]); print(r["citations"], r["abstained"], r["latency_s"], "s")

{"answer": The current hotel cap for Lisbon is 160 EUR per night for stays from 1 April 2026 onwards, due to rising hotel prices. Prior to that date, the cap remains at 140 EUR per night., "citations": ["PT-2026-02", "5.2"], "abstained": false}
[] False 13.35 s


In [7]:
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm

rows = []
for q in tqdm(questions):
    r = answer_question(q["question"], retrieved[q["id"]], llm)
    rows.append({"id": q["id"], "category": q["category"], "question": q["question"],
                 "expected_answer": q["expected_answer"], "sources": q["sources"], **r})

out = Path("results") / f"{datetime.now():%Y-%m-%d_%H%M}_{RUN_NAME}"
out.mkdir(parents=True, exist_ok=True)
with open(out / "answers.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
json.dump({"generator": GEN_MODEL, "top_k": TOP_K, "retriever": "8_hybrid (exclude_superseded, with_path, figures)"},
          open(out / "run_config.json", "w"), indent=2)
print("Sačuvano u", out, "| parse greške:", sum(r["parse_error"] for r in rows))

  0%|          | 0/49 [00:00<?, ?it/s]

Sačuvano u results/2026-10-14_2237_gen_hybrid_qwen14b | parse greške: 4


In [8]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/northstar_results
!cp -r {out} /content/drive/MyDrive/northstar_results/
print("Kopirano u Drive:", out.name)

Mounted at /content/drive
Kopirano u Drive: 2026-10-14_2237_gen_hybrid_qwen14b


In [9]:
!zip -qr results_gen.zip {out}
from google.colab import files
files.download("results_gen.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>